# 7) Finalize Core ATLAS Results

This notebook creates the curated result layer distributed with the SCO ATLAS. Only intrinsic per-complex quantities selected below are registered. Dataset-level analyses, correlations, rankings, family statistics, map positions and VNM decompositions remain reproducible outputs of the analysis notebooks and are not embedded in the System objects.

The notebook defaults to a CSV-only dry run. No ATLAS System is loaded or modified while `dry_run=True`.

In [ ]:
import os
import re
import sys

import scope

sys.path.insert(0, os.path.abspath('..'))
from atlas_functions import (
    base_refcode,
    read_csv,
    register_atlas_row_results,
    resolve_system_path,
    write_csv_rows,
)

## 1. Configuration

In [ ]:
data_path     = "/Users/sergivela/Documents/SCOPE/ATLAS/Data/"                    # Update with your path
analysis_path = "/Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/"  # Update with your path
env_path      = "/Users/sergivela/Documents/SCOPE/ATLAS/Data/scope_env_atlas.npy" # Update with your path

atlas_release          = "0.1-draft"
dry_run                = True
save_systems           = False
overwrite_atlas_results = False
write_manifest         = True
write_confirmation     = ""

data_path     = os.path.abspath(os.path.expanduser(data_path)) + os.sep
analysis_path = os.path.abspath(os.path.expanduser(analysis_path)) + os.sep
env_path      = os.path.abspath(os.path.expanduser(env_path))
outputs_path  = analysis_path + "Outputs/"

expected_scope_version = "0.9.6"
installed_scope_version = getattr(scope, "__version__", None)
if installed_scope_version != expected_scope_version:
    raise RuntimeError(f"This notebook requires SCOPE {expected_scope_version}; the active environment provides {installed_scope_version!r}.")
if save_systems and dry_run:
    raise ValueError("Set dry_run=False before enabling save_systems.")
if save_systems and write_confirmation != "WRITE ATLAS CORE RESULTS":
    raise ValueError("Set write_confirmation='WRITE ATLAS CORE RESULTS' before modifying System files.")

print(f"SCOPE version:    {installed_scope_version}")
print(f"ATLAS release:    {atlas_release}")
print(f"Dry run:          {dry_run}")
print(f"Save Systems:     {save_systems}")
print(f"Overwrite Results:{overwrite_atlas_results}")

## 2. Core Result Contract

The dictionaries below are the explicit release contract. Scalar values are stored as standard SCOPE `Data` objects in the `isolated` branch under `atlas_*` keys. The native workflow results remain untouched.

The initial contract contains molecular thermodynamics at 298.15 K, the computed molecular crossing temperature, basic composition labels, the optimized states used in the analysis, and selected local structural descriptors. The full-molecule convex-hull volume is intentionally excluded. Bibliographic metadata belongs to the Sources/System provenance rather than to the `isolated` branch.

In [ ]:
temperature = 298.15

thermodynamic_schema = [
    {'key': 'atlas_dHelec',      'column': 'dHelec[kJ/mol]',                'units': 'kj'},
    {'key': 'atlas_dHvib_298',   'column': 'dHvib[kJ/mol](T=298K)',        'units': 'kj', 'temperature': temperature},
    {'key': 'atlas_TdSvib_298',  'column': 'dSvib·T[kJ/mol](T=298K)',      'units': 'kj', 'temperature': temperature},
    {'key': 'atlas_TdSelec_298', 'column': 'dSelec·T[kJ/mol](T=298K)',     'units': 'kj', 'temperature': temperature},
    {'key': 'atlas_dGtot_298',   'column': 'dGtot[kJ/mol](T=298K)',        'units': 'kj', 'temperature': temperature},
    {'key': 'atlas_T12',         'column': 'T12[K]',                       'units': 'K'},
    {'key': 'atlas_homoleptic',  'column': 'homoleptic',                   'units': '', 'value_type': 'bool'},
    {'key': 'atlas_denticity',   'column': 'denticity',                    'units': '', 'value_type': 'str'},
]

structural_schema = [
    {'key': 'atlas_HS_minimum_state', 'column': 'HS_minimum_state',     'units': '', 'value_type': 'str'},
    {'key': 'atlas_LS_minimum_state', 'column': 'LS_minimum_state',     'units': '', 'value_type': 'str'},
    {'key': 'atlas_HS_dFeN',          'column': 'HS_dFeN',              'units': 'angstrom'},
    {'key': 'atlas_LS_dFeN',          'column': 'LS_dFeN',              'units': 'angstrom'},
    {'key': 'atlas_HS_Sigma',         'column': 'HS_Sigma',             'units': 'deg'},
    {'key': 'atlas_LS_Sigma',         'column': 'LS_Sigma',             'units': 'deg'},
    {'key': 'atlas_HS_CShM',          'column': 'HS_CShM',              'units': ''},
    {'key': 'atlas_LS_CShM',          'column': 'LS_CShM',              'units': ''},
    {'key': 'atlas_HS_FCS_volume',    'column': 'HS_FCS_CH_volume',     'units': 'angstrom^3'},
    {'key': 'atlas_LS_FCS_volume',    'column': 'LS_FCS_CH_volume',     'units': 'angstrom^3'},
]

print(f"Thermodynamic results: {len(thermodynamic_schema)}")
print(f"Structural results:    {len(structural_schema)}")

## 3. Validate Inputs and Prepare the Registration Plan

This stage reads only CSV files. It validates the selected columns and resolves each non-unique entry to its representative TMC. A future version of `nonunique_systems.csv` should provide `representative_refcode` explicitly; the current human-readable reason is supported as a temporary fallback.

In [ ]:
unique_path       = outputs_path + "unique_systems.csv"
nonunique_path    = outputs_path + "nonunique_systems.csv"
thermodynamic_path = outputs_path + "unique_thermodynamic_data.csv"
structural_path    = outputs_path + "unique_structural_data.csv"
manifest_path      = outputs_path + "atlas_core_results_manifest.csv"

unique_rows        = read_csv(unique_path)
nonunique_rows     = read_csv(nonunique_path)
thermodynamic_rows = read_csv(thermodynamic_path)
structural_rows    = read_csv(structural_path)

unique_by_refcode = {}
for row in unique_rows:
    unique_by_refcode[base_refcode(row['refcode'])] = row

thermodynamic_by_refcode = {}
for row in thermodynamic_rows:
    thermodynamic_by_refcode[base_refcode(row['refcode'])] = row

structural_by_refcode = {}
for row in structural_rows:
    structural_by_refcode[base_refcode(row['id'])] = row

required_thermodynamic_columns = {item['column'] for item in thermodynamic_schema}
required_structural_columns    = {item['column'] for item in structural_schema}
available_thermodynamic_columns = set(thermodynamic_rows[0]) if thermodynamic_rows else set()
available_structural_columns    = set(structural_rows[0]) if structural_rows else set()
missing_thermodynamic_columns = required_thermodynamic_columns - available_thermodynamic_columns
missing_structural_columns    = required_structural_columns - available_structural_columns
if missing_thermodynamic_columns:
    raise KeyError(f"Missing thermodynamic columns: {sorted(missing_thermodynamic_columns)}")
if missing_structural_columns:
    raise KeyError(f"Missing structural columns: {sorted(missing_structural_columns)}")

registration_plan = []
for refcode, row in unique_by_refcode.items():
    registration_plan.append({
        'entry_refcode': refcode,
        'canonical_refcode': refcode,
        'path': row['path'],
        'inherited': False,
    })

legacy_representative_rows = 0
for row in nonunique_rows:
    entry_refcode = base_refcode(row['refcode'])
    representative_refcode = base_refcode(row.get('representative_refcode', ''))
    if not representative_refcode:
        match = re.search(r'same as system ([A-Za-z]{6})', row.get('reason', ''), re.IGNORECASE)
        if match is None:
            raise ValueError(f"Cannot resolve the representative of {entry_refcode}")
        representative_refcode = match.group(1).upper()
        legacy_representative_rows += 1
    registration_plan.append({
        'entry_refcode': entry_refcode,
        'canonical_refcode': representative_refcode,
        'path': row['path'],
        'inherited': True,
    })

registration_plan.sort(key=lambda row: row['entry_refcode'])
missing_thermodynamic_rows = sorted({row['canonical_refcode'] for row in registration_plan} - set(thermodynamic_by_refcode))
missing_structural_rows    = sorted({row['canonical_refcode'] for row in registration_plan} - set(structural_by_refcode))
if missing_thermodynamic_rows:
    raise ValueError(f"Representatives missing thermodynamic data: {missing_thermodynamic_rows}")
if missing_structural_rows:
    raise ValueError(f"Representatives missing structural data: {missing_structural_rows}")

num_inherited = sum(row['inherited'] for row in registration_plan)
print(f"Representative systems: {len(registration_plan) - num_inherited}")
print(f"Inherited systems:      {num_inherited}")
print(f"Total planned systems:  {len(registration_plan)}")
print(f"Legacy reason fallback: {legacy_representative_rows}")

## 4. Register Core Results

With `dry_run=True`, this section reports the planned operation and exits without loading any System. With `dry_run=False` and `save_systems=False`, Systems are loaded and modified only in memory. Persistent writes require the explicit confirmation configured above.

In [ ]:
manifest_rows = []
registration_errors = []

if dry_run:
    print("Dry run complete: CSV schemas and representative mappings are valid.")
    print("No ATLAS System was loaded or modified.")
else:
    env = scope.load_binary(env_path)
    for index, planned in enumerate(registration_plan):
        entry_refcode     = planned['entry_refcode']
        canonical_refcode = planned['canonical_refcode']
        system_file       = resolve_system_path(planned['path'], data_path)
        try:
            system = scope.load_binary(system_file)
            system.set_paths_from_environment(env)
            exists, isolated = system.find_branch('isolated')
            if not exists:
                raise ValueError(f"{entry_refcode} has no isolated branch")

            thermodynamic_keys = register_atlas_row_results(
                isolated,
                thermodynamic_by_refcode[canonical_refcode],
                thermodynamic_schema,
                atlas_release=atlas_release,
                entry_refcode=entry_refcode,
                canonical_refcode=canonical_refcode,
                source_table=os.path.basename(thermodynamic_path),
                inherited=planned['inherited'],
                overwrite=overwrite_atlas_results,
            )
            structural_keys = register_atlas_row_results(
                isolated,
                structural_by_refcode[canonical_refcode],
                structural_schema,
                atlas_release=atlas_release,
                entry_refcode=entry_refcode,
                canonical_refcode=canonical_refcode,
                source_table=os.path.basename(structural_path),
                inherited=planned['inherited'],
                overwrite=overwrite_atlas_results,
            )
            registered_keys = thermodynamic_keys + structural_keys
            missing_keys = [key for key in registered_keys if key not in isolated.results]
            if missing_keys:
                raise RuntimeError(f"Results were not registered: {missing_keys}")

            if save_systems:
                system.save(system_file)
                status = 'saved'
            else:
                status = 'validated_in_memory'

            manifest_rows.append({
                'entry_refcode': entry_refcode,
                'canonical_refcode': canonical_refcode,
                'inherited': planned['inherited'],
                'system_file': planned['path'],
                'num_core_results': len(registered_keys),
                'atlas_release': atlas_release,
                'status': status,
                'error': '',
            })
            print(f"[{index + 1}/{len(registration_plan)}] {entry_refcode}: {status}")

        except Exception as exc:
            error = {
                'entry_refcode': entry_refcode,
                'canonical_refcode': canonical_refcode,
                'inherited': planned['inherited'],
                'system_file': planned['path'],
                'num_core_results': 0,
                'atlas_release': atlas_release,
                'status': 'error',
                'error': str(exc),
            }
            registration_errors.append(error)
            manifest_rows.append(error)
            print(f"[{index + 1}/{len(registration_plan)}] {entry_refcode}: ERROR - {exc}")

## 5. Export and Audit Manifest

In [ ]:
manifest_columns = [
    'entry_refcode', 'canonical_refcode', 'inherited', 'system_file',
    'num_core_results', 'atlas_release', 'status', 'error',
]

if not dry_run and write_manifest:
    write_csv_rows(manifest_path, manifest_rows, manifest_columns)
    print(f"Manifest written to: {manifest_path}")

if registration_errors:
    raise RuntimeError(f"Core-result registration failed for {len(registration_errors)} systems.")
if not dry_run:
    print(f"Registered Systems: {len(manifest_rows)}")
    print(f"Persisted Systems:  {sum(row['status'] == 'saved' for row in manifest_rows)}")